In [2]:
import pandas as pd
import numpy as np

# 1. Load raw datasets
courses = pd.read_csv("courses.csv")
assessments = pd.read_csv("assessments.csv")
student_info = pd.read_csv("studentInfo.csv")
student_reg = pd.read_csv("studentRegistration.csv")
student_assessment = pd.read_csv("studentAssessment.csv")
student_vle = pd.read_csv("studentVle.csv")
vle = pd.read_csv("vle.csv")

# Display confirmation and preview primary table
print("Datasets loaded successfully.")
student_info.head()

Datasets loaded successfully.


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass


Step 2: Extract & Aggregate Academic Performance Metrics
Merge assessment records with weighting data to calculate each student's total weighted score and completed assessments.

In [3]:
# Join student submissions with assessment weighting details
student_scores = pd.merge(student_assessment, assessments, on="id_assessment")

# Compute weighted contribution per assessment
student_scores["weighted_score"] = (student_scores["score"] * student_scores["weight"]) / 100

# Aggregate total score and assessment count per student module registration
academic_perf = student_scores.groupby(["id_student", "code_module", "code_presentation"]).agg(
    total_weighted_score=("weighted_score", "sum"),
    assessments_completed=("id_assessment", "count")
).reset_index()

# Preview aggregated academic performance
academic_perf.head()

,id_student,code_module,code_presentation,total_weighted_score,assessments_completed
0,6516,AAA,2014J,63.50,5
1,8462,DDD,2013J,34.90,3
2,8462,DDD,2014J,43.00,4
3,11391,AAA,2013J,82.40,5
4,23629,BBB,2013B,16.69,4


Step 3: Extract & Aggregate VLE Learning Engagement Metrics
Summarize online portal interaction logs into total click volume and distinct active learning days.

In [4]:
# Aggregate online activity logs from studentVle
vle_engagement = student_vle.groupby(["id_student", "code_module", "code_presentation"]).agg(
    total_clicks=("sum_click", "sum"),
    active_days=("date", "nunique")
).reset_index()

# Preview aggregated VLE engagement features
vle_engagement.head()

,id_student,code_module,code_presentation,total_clicks,active_days
0,6516,AAA,2014J,2791,159
1,8462,DDD,2013J,646,56
2,8462,DDD,2014J,10,1
3,11391,AAA,2013J,934,40
4,23629,BBB,2013B,161,16


Step 4: Build Master Integrated Dataset
Merge demographic profiles, enrollment records, academic scores, and engagement logs into a unified table for analytical modeling.



In [8]:
# 1. Start with demographic baseline and enrollment history
df_master = pd.merge(student_info, student_reg, on=["id_student", "code_module", "code_presentation"], how="left")

# 2. Merge performance and online engagement metrics
df_master = pd.merge(df_master, academic_perf, on=["id_student", "code_module", "code_presentation"], how="left")
df_master = pd.merge(df_master, vle_engagement, on=["id_student", "code_module", "code_presentation"], how="left")

# 3. Impute non-active/missing values with 0
df_master["total_weighted_score"] = df_master["total_weighted_score"].fillna(0)
df_master["assessments_completed"] = df_master["assessments_completed"].fillna(0)
df_master["total_clicks"] = df_master["total_clicks"].fillna(0)
df_master["active_days"] = df_master["active_days"].fillna(0)

# 4. Remove tracking keys irrelevant to ML features
df_master = df_master.drop(columns=["id_student", "code_module", "code_presentation"])

# Preview merged master analytical table
df_master.head()

,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,total_weighted_score,assessments_completed,total_clicks,active_days
0,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,-159.0,NaN,82.4,5.0,934.0,40.0
1,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,-53.0,NaN,65.4,5.0,1435.0,80.0
2,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,-92.0,12.0,0.0,0.0,281.0,12.0
3,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,-52.0,NaN,76.3,5.0,2158.0,123.0
4,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,-176.0,NaN,55.0,5.0,1034.0,70.0


Step 1: Check Data Structure & Missing Values
Inspect missing values and data types across the merged dataset.

In [9]:
# Missing values count and data types breakdown
eda_info = pd.DataFrame({
    "Missing Values": df_master.isnull().sum(),
    "Data Type": df_master.dtypes
})

# Display top fields with missing data checks
eda_info.head(10)

,Missing Values,Data Type
gender,0,object
region,0,object
highest_education,0,object
imd_band,1111,object
age_band,0,object
num_of_prev_attempts,0,int64
studied_credits,0,int64
disability,0,object
final_result,0,object
date_registration,45,float64


In [10]:
# ==========================================
# SEPARATE STEP: CLEANING MISSING VALUES
# ==========================================

# 1. Fill missing categorical socio-economic data with 'Unknown'
df_master["imd_band"] = df_master["imd_band"].fillna("Unknown")

# 2. Fill missing numerical registration dates with the median registration day
median_reg_date = df_master["date_registration"].median()
df_master["date_registration"] = df_master["date_registration"].fillna(median_reg_date)

# 3. Fill missing performance and engagement metrics with 0 (for inactive students)
df_master["total_weighted_score"] = df_master["total_weighted_score"].fillna(0)
df_master["assessments_completed"] = df_master["assessments_completed"].fillna(0)
df_master["total_clicks"] = df_master["total_clicks"].fillna(0)
df_master["active_days"] = df_master["active_days"].fillna(0)

# ==========================================
# VERIFICATION & DISPLAY OUTPUT
# ==========================================

# Check that missing values have been resolved
missing_summary = pd.DataFrame({
    "Missing Values After Cleaning": df_master[
        ["gender", "region", "highest_education", "imd_band", "age_band", 
         "num_of_prev_attempts", "studied_credits", "disability", 
         "final_result", "date_registration"]
    ].isnull().sum(),
    "Data Type": df_master[
        ["gender", "region", "highest_education", "imd_band", "age_band", 
         "num_of_prev_attempts", "studied_credits", "disability", 
         "final_result", "date_registration"]
    ].dtypes
})

# Display summary table ending with .head()
missing_summary.head(10)

,Missing Values After Cleaning,Data Type
gender,0,object
region,0,object
highest_education,0,object
imd_band,0,object
age_band,0,object
num_of_prev_attempts,0,int64
studied_credits,0,int64
disability,0,object
final_result,0,object
date_registration,0,float64


Step 2: Target Variable Distribution (final_result)
Examine the distribution of target outcomes (Pass, Fail, Withdrawn, Distinction) to check for class imbalance.

In [11]:
# Calculate counts and percentages of student outcomes
outcome_dist = df_master["final_result"].value_counts().reset_index()
outcome_dist.columns = ["Outcome", "Student Count"]
outcome_dist["Percentage (%)"] = (outcome_dist["Student Count"] / len(df_master) * 100).round(2)

# Display target class distribution
outcome_dist.head()

,Outcome,Student Count,Percentage (%)
0,Pass,12361,37.93
1,Withdrawn,10156,31.16
2,Fail,7052,21.64
3,Distinction,3024,9.28


Step 3: Analyze VLE Engagement vs. Academic Outcome
Compare online platform activity (total_clicks and active_days) across each student outcome category.



In [12]:
# Average online engagement broken down by final result
engagement_summary = df_master.groupby("final_result").agg(
    avg_total_clicks=("total_clicks", "mean"),
    avg_active_days=("active_days", "mean"),
    avg_weighted_score=("total_weighted_score", "mean")
).round(2).reset_index()

# Display engagement summary per outcome
engagement_summary.head()

,final_result,avg_total_clicks,avg_active_days,avg_weighted_score
0,Distinction,2666.76,110.04,100.60
1,Fail,651.85,33.18,27.07
2,Pass,1921.81,87.04,81.45
3,Withdrawn,313.95,16.29,7.75


Step 4: Demographic Impact on Student Success
Analyze how specific demographic factors (e.g., highest_education, disability, imd_band) correlate with student pass/fail rates.

In [13]:
# Outcome counts grouped by highest education level
education_analysis = pd.crosstab(
    df_master["highest_education"], 
    df_master["final_result"], 
    normalize="index"
).round(4) * 100

# Display education level breakdown
education_analysis.head()

final_result,Distinction,Fail,Pass,Withdrawn
highest_education,,,,
A Level or Equivalent,10.65,19.27,41.38,28.69
HE Qualification,14.74,16.70,41.44,27.12
Lower Than A Level,5.53,26.04,33.33,35.11
No Formal quals,4.61,27.38,25.07,42.94
Post Graduate Qualification,28.12,10.86,37.38,23.64


Step 5: Data Preprocessing & Model Training
Encode target variables (final_result), scale numeric fields, hot-encode categorical features, train the Logistic Regression classifier, and export the .pkl artifacts.

In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ==========================================
# 1. SEPARATE FEATURES & TARGET
# ==========================================

# Drop target and any potential target-leakage columns
columns_to_drop = ["final_result"]
if "date_unregistration" in df_master.columns:
    columns_to_drop.append("date_unregistration")

X = df_master.drop(columns=columns_to_drop)
y = df_master["final_result"]

# Encode target outcomes (Distinction, Pass, Fail, Withdrawn)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Identify numerical and categorical features
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# ==========================================
# 2. BUILD ROBUST PREPROCESSING PIPELINES
# ==========================================

# Pipeline for numerical features: Fill missing values with median, then scale
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical features: Fill missing values with 'Unknown', then One-Hot Encode
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine numerical and categorical pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)

# ==========================================
# 3. TRAIN / TEST SPLIT & MODEL FIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Fit preprocessor on training data and transform both sets
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train multi-class Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_processed, y_train)

# ==========================================
# 4. EVALUATION & ARTIFACT EXPORT
# ==========================================

y_pred = model.predict(X_test_processed)

# Print performance report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Save artifacts for Streamlit app
joblib.dump(model, "logistic_regression_model.pkl")
joblib.dump(preprocessor, "student_preprocessor.pkl")
joblib.dump(label_encoder, "target_label_encoder.pkl")

# Preview actual vs predicted outcomes
results_preview = pd.DataFrame({
    "Actual Target": label_encoder.inverse_transform(y_test[:5]),
    "Predicted Target": label_encoder.inverse_transform(y_pred[:5])
})

results_preview.head()

Classification Report:

              precision    recall  f1-score   support

 Distinction       0.67      0.06      0.10       605
        Fail       0.47      0.27      0.34      1411
        Pass       0.69      0.92      0.79      2472
   Withdrawn       0.72      0.86      0.78      2031

    accuracy                           0.68      6519
   macro avg       0.64      0.52      0.51      6519
weighted avg       0.65      0.68      0.63      6519



,Actual Target,Predicted Target
0,Fail,Withdrawn
1,Pass,Pass
2,Fail,Withdrawn
3,Withdrawn,Withdrawn
4,Fail,Pass


In [16]:
# Train multi-class Logistic Regression with class weighting
model = LogisticRegression(
    max_iter=1000, 
    class_weight="balanced", 
    C=1.0, 
    solver="lbfgs"
)

model.fit(X_train_processed, y_train)

# Evaluation
y_pred = model.predict(X_test_processed)

print("Tuned Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Overwrite existing model artifacts with tuned version
joblib.dump(model, "logistic_regression_model.pkl")
joblib.dump(preprocessor, "student_preprocessor.pkl")
joblib.dump(label_encoder, "target_label_encoder.pkl")
print("Tuned model saved successfully!")

Tuned Classification Report:

              precision    recall  f1-score   support

 Distinction       0.29      0.64      0.40       605
        Fail       0.41      0.41      0.41      1411
        Pass       0.72      0.49      0.58      2472
   Withdrawn       0.75      0.78      0.77      2031

    accuracy                           0.58      6519
   macro avg       0.54      0.58      0.54      6519
weighted avg       0.62      0.58      0.59      6519

Tuned model saved successfully!


In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==========================================
# 1. EVALUATE LOGISTIC REGRESSION MODEL
# ==========================================

lr_pred = model.predict(X_test_processed)
lr_acc = accuracy_score(y_test, lr_pred)

print(f"==========================================")
print(f"LOGISTIC REGRESSION OVERALL ACCURACY: {lr_acc * 100:.2f}%")
print(f"==========================================\n")

# ==========================================
# 2. TRAIN & EVALUATE RANDOM FOREST (ADVANCED)
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_processed, y_train)
rf_pred = rf_model.predict(X_test_processed)
rf_acc = accuracy_score(y_test, rf_pred)

print(f"==========================================")
print(f"RANDOM FOREST OVERALL ACCURACY: {rf_acc * 100:.2f}%")
print(f"==========================================\n")

# ==========================================
# 3. DETAILED COMPARISON TABLE
# ==========================================

comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression (Current)", "Random Forest (Advanced)"],
    "Overall Accuracy": [f"{lr_acc * 100:.2f}%", f"{rf_acc * 100:.2f}%"],
    "Distinction Recall": [
        f"{classification_report(y_test, lr_pred, output_dict=True)['0']['recall']*100:.1f}%",
        f"{classification_report(y_test, rf_pred, output_dict=True)['0']['recall']*100:.1f}%"
    ],
    "Fail Recall": [
        f"{classification_report(y_test, lr_pred, output_dict=True)['1']['recall']*100:.1f}%",
        f"{classification_report(y_test, rf_pred, output_dict=True)['1']['recall']*100:.1f}%"
    ]
})

comparison_df.head()

LOGISTIC REGRESSION OVERALL ACCURACY: 57.72%

RANDOM FOREST OVERALL ACCURACY: 69.17%



,Model,Overall Accuracy,Distinction Recall,Fail Recall
0,Logistic Regression (Current),57.72%,64.0%,40.9%
1,Random Forest (Advanced),69.17%,69.3%,45.2%


In [19]:
import os
import warnings

# Set loky CPU count to silence the physical core warning
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # Replace 4 with your desired core count

# Suppress UserWarnings from joblib/loky
warnings.filterwarnings("ignore", category=UserWarning, module="joblib")


import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

import xgboost as xgb
import lightgbm as lgb
import joblib

# Calculate sample weights to keep class balance for boosted algorithms
sample_weights_train = compute_sample_weight("balanced", y_train)

# ==========================================
# 1. XGBOOST CLASSIFIER
# ==========================================
xgb_model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss"
)

xgb_model.fit(X_train_processed, y_train, sample_weight=sample_weights_train)
xgb_pred = xgb_model.predict(X_test_processed)
xgb_acc = accuracy_score(y_test, xgb_pred)

# ==========================================
# 2. LIGHTGBM CLASSIFIER
# ==========================================
lgb_model = lgb.LGBMClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42,
    verbosity=-1
)

lgb_model.fit(X_train_processed, y_train)
lgb_pred = lgb_model.predict(X_test_processed)
lgb_acc = accuracy_score(y_test, lgb_pred)

# ==========================================
# 3. COMPREHENSIVE BENCHMARK TABLE
# ==========================================
benchmark_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"],
    "Overall Accuracy": [
        f"{lr_acc * 100:.2f}%", 
        f"{rf_acc * 100:.2f}%", 
        f"{xgb_acc * 100:.2f}%", 
        f"{lgb_acc * 100:.2f}%"
    ],
    "Distinction Recall": [
        f"{classification_report(y_test, lr_pred, output_dict=True)['0']['recall']*100:.1f}%",
        f"{classification_report(y_test, rf_pred, output_dict=True)['0']['recall']*100:.1f}%",
        f"{classification_report(y_test, xgb_pred, output_dict=True)['0']['recall']*100:.1f}%",
        f"{classification_report(y_test, lgb_pred, output_dict=True)['0']['recall']*100:.1f}%"
    ],
    "Fail Recall": [
        f"{classification_report(y_test, lr_pred, output_dict=True)['1']['recall']*100:.1f}%",
        f"{classification_report(y_test, rf_pred, output_dict=True)['1']['recall']*100:.1f}%",
        f"{classification_report(y_test, xgb_pred, output_dict=True)['1']['recall']*100:.1f}%",
        f"{classification_report(y_test, lgb_pred, output_dict=True)['1']['recall']*100:.1f}%"
    ]
})

benchmark_df.head()

,Model,Overall Accuracy,Distinction Recall,Fail Recall
0,Logistic Regression,57.72%,64.0%,40.9%
1,Random Forest,69.17%,69.3%,45.2%
2,XGBoost,70.35%,86.9%,54.2%
3,LightGBM,70.36%,86.1%,55.3%


In [20]:
import joblib

# Assign LightGBM as the winning production model (70.36% accuracy)
winning_model = lgb_model  

# Export production artifacts
joblib.dump(winning_model, "logistic_regression_model.pkl")  # Kept filename constant for app loading
joblib.dump(preprocessor, "student_preprocessor.pkl")
joblib.dump(label_encoder, "target_label_encoder.pkl")

print("LightGBM model (70.36% accuracy) successfully saved for Streamlit Cloud deployment!")

LightGBM model (70.36% accuracy) successfully saved for Streamlit Cloud deployment!
